In [26]:
!pip install -q sentence-transformers scikit-learn gradio pandas
# Install
!pip install sentence-transformers torch

In [27]:

from sentence_transformers import SentenceTransformer
import torch
import torch.nn as nn
import torch.optim as optim

In [28]:
# Load embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [29]:
texts = [
    "Write a professional email",
    "Fix Python bug",
    "Explain transformers",
    "Create a creative story",
    "How does machine learning work",
    "Summarize this research paper",
    "Generate SQL query",
    "Debug neural network",
    "Explain attention mechanism",
    "Write marketing content",
    "Translate English to Sanskrit",
    "Create AI architecture",
    "Build RAG pipeline",
    "Explain vector databases",
    "Generate meeting summary",
    "Design REST API",
    "Optimize Python code",
    "Create chatbot workflow",
    "Explain embeddings",
    "Write product description"
]

In [30]:
# Create embeddings
embeddings = model.encode(texts)


In [31]:
 #Convert to tensor
X = torch.tensor(embeddings, dtype=torch.float32)


In [32]:
# Autoencoder
class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(384, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 384)
        )

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

# Initialize
autoencoder = Autoencoder()

criterion = nn.MSELoss()
optimizer = optim.Adam(autoencoder.parameters(), lr=0.001)

In [33]:
# Train
for epoch in range(100):

    reconstructed = autoencoder(X)

    loss = criterion(reconstructed, X)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

print("Training complete")

Epoch 0, Loss: 0.006433440838009119
Epoch 50, Loss: 0.001187870861031115
Training complete


In [34]:
# ============================================
# TEST THE AUTOENCODER
# ============================================

# Disable gradients for testing
with torch.no_grad():

    # Get latent vectors
    latent_vectors = autoencoder.encoder(X)

    # Reconstruct embeddings
    reconstructed_embeddings = autoencoder.decoder(latent_vectors)

# Print latent space shape
print("Latent Shape:")
print(latent_vectors.shape)

# Print one latent vector
print("\nSample Latent Vector:")
print(latent_vectors[0])

# Compare original vs reconstructed
print("\nOriginal Embedding (first 10 values):")
print(X[0][:10])

print("\nReconstructed Embedding (first 10 values):")
print(reconstructed_embeddings[0][:10])

# Reconstruction error
mse = torch.mean((X - reconstructed_embeddings) ** 2)

print("\nReconstruction Error:")
print(mse.item())

Latent Shape:
torch.Size([20, 32])

Sample Latent Vector:
tensor([ 0.0859,  0.2535,  0.2173,  0.2347,  0.0848, -0.2180, -0.1197,  0.3051,
        -0.1139,  0.2227,  0.0709, -0.1446,  0.3084, -0.1741,  0.0964, -0.0670,
         0.1071,  0.0445, -0.0435, -0.0189,  0.3507,  0.1045,  0.1562, -0.1297,
         0.1244,  0.3041, -0.0028, -0.0920,  0.1228, -0.1455, -0.0765, -0.2039])

Original Embedding (first 10 values):
tensor([-0.0188,  0.0499,  0.0646,  0.0159,  0.0539, -0.0404,  0.0249,  0.0640,
        -0.0040,  0.0127])

Reconstructed Embedding (first 10 values):
tensor([-0.0192,  0.0429,  0.0488,  0.0157,  0.0551, -0.0392,  0.0270,  0.0664,
        -0.0032,  0.0124])

Reconstruction Error:
6.521461909869686e-05
